# Predictive Maintenance Model for Textile MachineryTrains Random Forest and XGBoost classifiers to predict machine failure from sensor data. Includes proper evaluation beyond plain accuracy, since the failure class is rare (~3.4% of records).

In [ ]:
import os# Set your Kaggle API token using Colab Secrets (recommended) or environment variable.# NEVER hardcode your token directly in a notebook that will be pushed to GitHub.# In Colab: use the key icon in the left sidebar to add KAGGLE_API_TOKEN as a secret, then:# from google.colab import userdata# os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')import pandas as pd# 2. Download the Predictive Maintenance Dataset!kaggle datasets download -d shivamb/machine-predictive-maintenance-classification# 3. Unzip the file!unzip -o -q machine-predictive-maintenance-classification.zip# 4. Load the data into a Pandas DataFramedf = pd.read_csv('predictive_maintenance.csv')print(f"Success! We have {len(df)} machine sensor records loaded.")print("Here is a sneak peek at the data:")df.head()

In [ ]:
import pandas as pdfrom sklearn.model_selection import train_test_splitfrom sklearn.ensemble import RandomForestClassifierfrom xgboost import XGBClassifierfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrixfrom sklearn.preprocessing import LabelEncoderprint("Step 1: Engineering feature extraction pipeline...")X = df.drop(['UDI', 'Product ID', 'Target', 'Failure Type'], axis=1)y = df['Target']  # 0 = No Failure, 1 = Machine Failure# Clean column names so XGBoost doesn't crash on special charactersX.columns = X.columns.str.replace(r'[\[\]<]', '', regex=True)# Machine 'Type' is text (L, M, H) - encode it numericallyle = LabelEncoder()X['Type'] = le.fit_transform(X['Type'])# Stratified split keeps the same failure ratio in train and test setsX_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)print("Step 2: Training Random Forest Model...")rf_model = RandomForestClassifier(n_estimators=100, random_state=42)rf_model.fit(X_train, y_train)rf_preds = rf_model.predict(X_test)rf_acc = accuracy_score(y_test, rf_preds) * 100print("Step 3: Training XGBoost Model...")xgb_model = XGBClassifier(eval_metric='logloss', random_state=42)xgb_model.fit(X_train, y_train)xgb_preds = xgb_model.predict(X_test)xgb_acc = accuracy_score(y_test, xgb_preds) * 100print(f"\nRandom Forest Accuracy: {rf_acc:.2f}%")print(f"XGBoost Accuracy:       {xgb_acc:.2f}%")

## Why Accuracy Alone Is Misleading HereThe failure class is rare (~3.4% of records). A model predicting 'No Failure' every time would already score ~96-97% accuracy without being useful. Precision and recall on the failure class are the metrics that actually matter.

In [ ]:
print("--- CLASS BALANCE ---")print(y.value_counts())baseline_acc = (y_test.value_counts().max() / len(y_test)) * 100print(f"\nA model predicting 'No Failure' for every row would score: {baseline_acc:.2f}% accuracy")print("Compare this to the accuracy numbers above - that's the real bar to beat.\n")print("--- RANDOM FOREST: FULL REPORT ---")print(classification_report(y_test, rf_preds, target_names=['No Failure', 'Failure']))print("Confusion Matrix:")print(confusion_matrix(y_test, rf_preds))print("\n--- XGBOOST: FULL REPORT ---")print(classification_report(y_test, xgb_preds, target_names=['No Failure', 'Failure']))print("Confusion Matrix:")print(confusion_matrix(y_test, xgb_preds))

## Feature ImportanceWhich sensor readings matter most for predicting failure, according to the XGBoost model.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as snsimport pandas as pdimportances = xgb_model.feature_importances_features = X.columnsimportance_df = pd.DataFrame({'Feature': features, 'Importance': importances})importance_df = importance_df.sort_values(by='Importance', ascending=False)plt.figure(figsize=(10, 6))sns.barplot(x='Importance', y='Feature', data=importance_df, hue='Feature', palette='viridis', legend=False)plt.title('Sensor Importance in Predicting Machine Failure (XGBoost)', fontsize=14, fontweight='bold')plt.xlabel('Importance Score (Impact on Breakdown)', fontsize=12)plt.ylabel('Machine Sensor Metrics', fontsize=12)sns.despine()plt.tight_layout()plt.savefig('feature_importance.png', dpi=120)plt.show()

## Key Findings1. **XGBoost outperforms Random Forest** on the failure class: 75% recall and 91% precision (vs. Random Forest's 65% recall and 88% precision).2. **XGBoost catches 51 of 68 real failures** in the test set, with only 5 false positives out of 1,932 healthy machines.3. **Torque and Air Temperature are the top two failure-driving features**, guiding which sensors deserve the most monitoring attention.4. This is snapshot sensor data (10,000 independent readings), not a time-series — there is no timestamp or sequential dependency between rows.